In [ ]:
# ==============================================================
# TDS Demo: NCSS + WMS Services
# Requires: siphon, xarray, matplotlib, cartopy, IPython
# pip install siphon xarray matplotlib cartopy
# ==============================================================

# ── Cell 1 ── Imports ──────────────────────────────────────────
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from io import BytesIO
from datetime import datetime

from siphon.catalog import TDSCatalog
from siphon.ncss import NCSS

# ── Cell 2 ── Connect to TDS catalog ──────────────────────────
# Using Unidata's public TDS as example; swap for your own URL
CATALOG_URL = (
    "https://thredds.ucar.edu/thredds/catalog/"
    "grib/NCEP/GFS/Global_0p25deg/catalog.xml"
)

catalog = TDSCatalog(CATALOG_URL)
print("Available datasets:")
for name in list(catalog.datasets)[:5]:   # show first 5
    print(" •", name)

# Grab the latest dataset
ds_name = list(catalog.datasets)[0]
dataset = catalog.datasets[ds_name]
print(f"\nUsing: {ds_name}")

# ── Cell 3 ── Inspect available services ──────────────────────
print("Services on this dataset:")
for svc in dataset.access_urls:
    print(f"  {svc:20s} → {dataset.access_urls[svc]}")

# ── Cell 4 ── WMS — Build a GetMap URL and display the tile ───
from IPython.display import Image, display

wms_base = dataset.access_urls["WMS"]

# Construct a WMS GetMap request (lat/lon bounding box, 800×400 px)
params = dict(
    service    = "WMS",
    version    = "1.3.0",
    request    = "GetMap",
    layers     = "Temperature_isobaric",      # adjust to variable name
    styles     = "boxfill/rainbow",
    crs        = "CRS:84",
    bbox       = "-180,-90,180,90",           # full globe
    width      = 800,
    height     = 400,
    format     = "image/png",
    colorscalerange = "220,310",              # K range
    time       = "2024-01-01T00:00:00Z",      # adjust or leave out for latest
)

import requests
from urllib.parse import urlencode

wms_url = wms_base + "?" + urlencode(params)
print("WMS URL:\n", wms_url)

# Fetch and display inline
resp = requests.get(wms_url, timeout=30)
resp.raise_for_status()

img = mpimg.imread(BytesIO(resp.content))
fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(img, extent=[-180, 180, -90, 90], aspect="auto")
ax.set_title("WMS — Temperature at Isobaric Level (GetMap)", fontsize=13)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

# ── Cell 5 ── WMS — GetCapabilities (inspect layers) ──────────
import xml.etree.ElementTree as ET

cap_url = wms_base + "?service=WMS&version=1.3.0&request=GetCapabilities"
cap_resp = requests.get(cap_url, timeout=30)
root = ET.fromstring(cap_resp.content)
ns = {"wms": "http://www.opengis.net/wms"}

print("WMS Layers available (first 10):")
for layer in root.findall(".//wms:Layer/wms:Name", ns)[:10]:
    print(" •", layer.text)

# ── Cell 6 ── NCSS — Subset by variable, bbox, and time ───────
ncss_url = dataset.access_urls["NetcdfSubset"]
ncss = NCSS(ncss_url)

print("NCSS variables (first 10):")
for v in list(ncss.variables)[:10]:
    print(" •", v)

# ── Cell 7 ── NCSS — Point/box query and download ─────────────
query = ncss.query()

query.variables(
    "Temperature_isobaric",
    "Geopotential_height_isobaric"
)

query.lonlat_box(
    north=50, south=20,
    east=-60, west=-130        # CONUS region
)

query.vertical_level(85000)    # 850 hPa in Pa

query.time_range(
    datetime(2024, 1, 1, 0),
    datetime(2024, 1, 1, 12)
)

query.accept("netcdf4")

# Download the subset into an xarray Dataset
data = ncss.get_data(query)
ds = xr.open_dataset(BytesIO(data.read()))
print("\nSubsetted dataset:")
print(ds)

# ── Cell 8 ── NCSS — Plot the subset ──────────────────────────
# Grab first time step of Temperature at 850 hPa
temp = ds["Temperature_isobaric"].isel(time=0).squeeze()

# Convert K → °C
temp_c = temp - 273.15

fig, ax = plt.subplots(figsize=(10, 6))
cf = ax.contourf(
    ds["lon"], ds["lat"], temp_c,
    levels=20, cmap="RdYlBu_r"
)
cs = ax.contour(
    ds["lon"], ds["lat"], temp_c,
    levels=10, colors="k", linewidths=0.4, alpha=0.5
)
ax.clabel(cs, fmt="%d°C", fontsize=8)
plt.colorbar(cf, ax=ax, label="Temperature (°C)")
ax.set_title("NCSS Subset — 850 hPa Temperature (CONUS)", fontsize=13)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

# ── Cell 9 ── NCSS — Time series at a single point ────────────
point_query = ncss.query()
point_query.variables("Temperature_isobaric")
point_query.lonlat_point(-105, 40)          # Boulder, CO
point_query.vertical_level(85000)
point_query.time_range(
    datetime(2024, 1, 1, 0),
    datetime(2024, 1, 2, 0)
)
point_query.accept("csv")

import pandas as pd
point_data = ncss.get_data(point_query)
df = pd.read_csv(BytesIO(point_data.read()))
df["time"] = pd.to_datetime(df["time"])
df["Temperature_isobaric"] -= 273.15       # K → °C

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["time"], df["Temperature_isobaric"], marker="o", linewidth=2)
ax.set_title("NCSS Point Query — 850 hPa Temperature, Boulder CO", fontsize=13)
ax.set_xlabel("Time (UTC)"); ax.set_ylabel("Temperature (°C)")
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# ── Cell 10 ── Summary printout ───────────────────────────────
print("=" * 55)
print("TDS Demo Summary")
print("=" * 55)
print(f"  Dataset   : {ds_name}")
print(f"  WMS map   : Temperature_isobaric, global")
print(f"  NCSS box  : CONUS, 850 hPa, 00Z–12Z Jan 1 2024")
print(f"  NCSS point: Boulder CO (-105, 40), 24-hr series")
print("=" * 55)